In [ ]:
# Optional: install required libraries for this lecture
%pip install -q openai


### Step 1: Enter your OpenRouter key and setup the client
We use OpenRouter's OpenAI-compatible API. Your key is entered securely and is not saved in the notebook.


In [3]:
import time
from getpass import getpass
from openai import OpenAI, RateLimitError, APITimeoutError

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    timeout=10,  # seconds
)

MODEL = "openai/gpt-4o-mini"  # Change to any OpenRouter-supported model
print("OpenRouter client ready")


Client ready


### Step 2: Simple retry with exponential backoff
We retry a few times on rate limit or timeout, sleeping 1s, 2s, 4s...


In [4]:
def chat_with_retry(prompt: str, retries: int = 3):
    delay = 1
    for attempt in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            return resp.choices[0].message.content
        except (RateLimitError, APITimeoutError) as e:
            if attempt == retries:
                raise
            print(f"Attempt {attempt} failed ({type(e).__name__}). Retrying in {delay}s...")
            time.sleep(delay)
            delay *= 2

print(chat_with_retry("Say a 5-word greeting."))


Hello! Hope you're having a great day!


### Step 3: Quick notes
- OpenRouter provides a common API endpoint for multiple model providers.
- Respect provider/model rate limits and handle `429` responses gracefully.
- Prefer server-side batching where possible.
- Keep timeouts reasonable (10–30s) and handle failures gracefully.
- Change `MODEL` above to try another OpenRouter-supported model.
